In [ ]:
# ===== 環境セットアップ =====
import os, subprocess

def _is_kaggle():
    return os.path.exists("/kaggle/input")

def _is_runpod():
    return os.environ.get("RUNPOD_POD_ID") is not None

ENV = "kaggle" if _is_kaggle() else "runpod" if _is_runpod() else "local"
print(f"[env] {ENV}")

if ENV == "runpod":
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import json as _json
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as _f:
        _json.dump({"username": os.environ["KAGGLE_USERNAME"],
                    "key": os.environ["KAGGLE_KEY"]}, _f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)


In [ ]:
# ===== パス定義 =====
import glob as _glob

def _find_path(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return candidates[0]

if ENV == "kaggle":
    MODEL_DIR    = _find_path(
                       "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
                       "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1")
    # CoT データセット（konbu17 または kienngx をマウント）
    COT_CSV      = _find_path(
                       "/kaggle/input/nemotron-sft-lora-cot-selection/train_split_with_cot.csv",
                       "/kaggle/input/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv")
    OUTPUT_DIR   = "/kaggle/working/adapter"
elif ENV == "runpod":
    MODEL_DIR    = "/workspace/models/nemotron-3-nano-30b-a3b-bf16"
    COT_CSV      = "/workspace/data/konbu17-cot/train_split_with_cot.csv"
    OUTPUT_DIR   = "/workspace/output/adapter"
else:  # local
    MODEL_DIR    = None
    COT_CSV      = "data/konbu17-cot/train_split_with_cot.csv"
    OUTPUT_DIR   = "output/adapter"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"MODEL_DIR  = {MODEL_DIR}  exists={os.path.exists(str(MODEL_DIR))}")
print(f"COT_CSV    = {COT_CSV}  exists={os.path.exists(str(COT_CSV))}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")


In [ ]:
# ===== ライブラリインストール =====
!pip install -q peft trl


In [ ]:
# ===== データ確認 =====
import pandas as pd

df = pd.read_csv(COT_CSV)
print(f"shape: {df.shape}")
print(f"columns: {df.columns.tolist()}")
print()
print("問題タイプ別件数:")
print(df['type'].value_counts())
print()
print("--- CoT サンプル ---")
print(df['generated_cot'].iloc[0][:400])


In [ ]:
# ===== 学習実行 =====
# ここのパラメータを変えて実験する

LORA_RANK  = 32
EPOCHS     = 2
LR         = 5e-5
BATCH_SIZE = 1
GRAD_ACCUM = 4
MAX_SEQ    = 2048
SUBSAMPLE  = None   # 動作確認時は 100 など小さい値に

!python train.py \
    --model_dir   "{MODEL_DIR}" \
    --data_csv    "{COT_CSV}" \
    --output_dir  "{OUTPUT_DIR}" \
    --lora_rank   {LORA_RANK} \
    --epochs      {EPOCHS} \
    --lr          {LR} \
    --batch_size  {BATCH_SIZE} \
    --grad_accum  {GRAD_ACCUM} \
    --max_seq_len {MAX_SEQ} \
    --zip_output \
    {f'--subsample {SUBSAMPLE}' if SUBSAMPLE else ''}


In [ ]:
# ===== 出力確認 =====
print("adapter files:", os.listdir(OUTPUT_DIR))

zip_path = os.path.join(os.path.dirname(OUTPUT_DIR), 'submission.zip')
if os.path.exists(zip_path):
    import zipfile
    with zipfile.ZipFile(zip_path) as zf:
        print("submission.zip contents:", zf.namelist())
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f"submission.zip size: {size_mb:.1f} MB")
